In [4]:
from thortils.navigation import get_shortest_path_to_object_type, get_shortest_path_to_object
from thortils.agent import thor_pose_as_tuple, thor_reachable_positions, thor_place_agent_randomly
import thortils as tt

In [5]:
import pandas as pd
from PIL import Image
from torch.nn import CosineSimilarity
from transformers import AutoImageProcessor, SwinModel, AutoProcessor, AutoModel
import torch
from IPython.display import clear_output
import os

cossim = CosineSimilarity(dim=1, eps=1e-6)
import cv2
import numpy as np
from ai2thor.controller import Controller
import matplotlib.pyplot as plt

from time import sleep
# Function to calculate points on the ellipse
def calculate_ellipse_points(center, axes, angle, num_points):
	points = []
	angle_rad = np.deg2rad(angle)
	
	for t in np.linspace(0, -np.pi, num_points):
		x = center[0] + axes[0] * np.cos(t) * np.cos(angle_rad) - axes[1] * np.sin(t) * np.sin(angle_rad)
		y = center[1] + axes[0] * np.cos(t) * np.sin(angle_rad) + axes[1] * np.sin(t) * np.cos(angle_rad)
		points.append((int(x), int(y)))
	return points


/Users/davidebuoso/miniforge3/envs/robothor/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
actions= [
	[{"action":"RotateRight", "degrees":45}], 
	[{"action":"RotateRight", "degrees":30}, {"action":"MoveAhead", "moveMagnitude":0.1}], 
	[{"action":"RotateRight", "degrees":15}, {"action":"MoveAhead", "moveMagnitude":0.1}],
	[{"action":"MoveAhead", "moveMagnitude":0.2}], 
	[{"action":"RotateLeft", "degrees":15}, {"action":"MoveAhead", "moveMagnitude":0.1}], 
	[{"action":"RotateLeft", "degrees":30}, {"action":"MoveAhead", "moveMagnitude":0.1}],
	[{"action":"RotateLeft", "degrees":45}], 
	[{"action":"LookUp"}], 
	[{"action":"LookDown"}], 
	[{"action":"Done"}]]

In [7]:

def rotate_compass(compass, direction):
	# Determine the shift value from the rotation_map
	shift = rotation_map.get(direction, 0)  # Default to 0 if command is not found
	# Rotate the compass list by the shift value
	shifted_compass = compass[-shift:] + compass[:-shift]
	return shifted_compass

rotation_map = {
	0: -3,
	1: -2,
	2: -1,
	3: 0,
   	4: 1,
	5: 2,
	6: 3
}

In [53]:
def init_controller():
    return tt.launch_controller(
    {"scene": "FloorPlan1", 
     "VISIBILITY_DISTANCE": 1,
     "IMAGE_WIDTH": 640,
     "IMAGE_HEIGHT": 480,
     "FOV": 63.45,
    }
)

In [54]:
controller = init_controller()


In [41]:

center = (controller.width//2, controller.height)  # Center of the ellipse
axes = (controller.width//2-30, controller.height*0.25)     # Major and minor axes lengths
angle = 0          # Rotation angle of the ellipse in degrees
num_points = 7
	# Number of points to generate on the perimeter
circle_radius = 5   # Radius of the circles to draw
# Calculate ellipse points
ellipse_points = calculate_ellipse_points(center, axes, angle, num_points)

In [42]:
kitchens = [f"FloorPlan{i}" for i in range(1, 21)]
living_rooms = [f"FloorPlan{200 + i}" for i in range(1, 21)]
bedrooms = [f"FloorPlan{300 + i}" for i in range(1, 21)]
bathrooms = [f"FloorPlan{400 + i}" for i in range(1, 21)]

kitchen_objects = ["HousePlant", "Sink", "DiningTable", "Knife", "Fridge", "Bowl", "Cabinet"]
living_room_objects = ["HousePlant", "CoffeeTable", "Cloth", "KeyChain", "WateringCan", "Bowl"]
bedroom_objects = ["Bed", "Lamp", "Book", "Chair", "LightSwitch", "SideTable", "Candle", "Painting", "Bowl"]
bathroom_objects = ["Sink", "Watch", "Cabinet", "LightSwitch", "Toilet", "Candle", "SprayBottle"]
all_objects = kitchen_objects + living_room_objects + bedroom_objects + bathroom_objects
all_objects

['HousePlant',
 'Sink',
 'DiningTable',
 'Knife',
 'Fridge',
 'Bowl',
 'Cabinet',
 'HousePlant',
 'CoffeeTable',
 'Cloth',
 'KeyChain',
 'WateringCan',
 'Bowl',
 'Bed',
 'Lamp',
 'Book',
 'Chair',
 'LightSwitch',
 'SideTable',
 'Candle',
 'Painting',
 'Bowl',
 'Sink',
 'Watch',
 'Cabinet',
 'LightSwitch',
 'Toilet',
 'Candle',
 'SprayBottle']

In [57]:
kitchens_obj_dict = dict()
kitchen_objects_per = {k:0 for k in kitchen_objects}
total_obj=0
for k in kitchens[:10]:
	controller.scene = k
	controller.reset()
	obj_set = [obj["objectType"] for obj in controller.last_event.metadata["objects"] if obj["objectType"] in kitchen_objects]
	if len(obj_set) > 0:
		obj_selected = list(set(obj_set))
		obj_keep=[]
		for o in obj_selected:
			if kitchen_objects_per[o] < 5:
				kitchen_objects_per[o]+=1
				obj_keep.append(o)
				
		obj_selected = obj_keep
		#kitchen_objects.remove(obj_selected)
		kitchens_obj_dict[k]=obj_selected
		total_obj+=len(obj_selected)
kitchens_obj_dict, total_obj

({'FloorPlan1': ['Knife', 'Sink', 'Fridge', 'HousePlant', 'Bowl', 'Cabinet'],
  'FloorPlan2': ['Knife', 'Sink', 'Fridge', 'Bowl', 'Cabinet'],
  'FloorPlan3': ['Knife', 'Sink', 'Fridge', 'HousePlant', 'Bowl', 'Cabinet'],
  'FloorPlan4': ['Knife',
   'Sink',
   'Fridge',
   'HousePlant',
   'Bowl',
   'Cabinet',
   'DiningTable'],
  'FloorPlan5': ['Knife', 'Sink', 'Fridge', 'HousePlant', 'Bowl', 'Cabinet'],
  'FloorPlan6': [],
  'FloorPlan7': ['HousePlant', 'DiningTable'],
  'FloorPlan8': [],
  'FloorPlan9': ['DiningTable'],
  'FloorPlan10': []},
 33)

In [58]:
to_remove = []
for k,v in kitchens_obj_dict.items():
    if len(v)==0:
        to_remove.append(k)
for k in to_remove:
	del kitchens_obj_dict[k]

In [59]:
kitchens_obj_dict

{'FloorPlan1': ['Knife', 'Sink', 'Fridge', 'HousePlant', 'Bowl', 'Cabinet'],
 'FloorPlan2': ['Knife', 'Sink', 'Fridge', 'Bowl', 'Cabinet'],
 'FloorPlan3': ['Knife', 'Sink', 'Fridge', 'HousePlant', 'Bowl', 'Cabinet'],
 'FloorPlan4': ['Knife',
  'Sink',
  'Fridge',
  'HousePlant',
  'Bowl',
  'Cabinet',
  'DiningTable'],
 'FloorPlan5': ['Knife', 'Sink', 'Fridge', 'HousePlant', 'Bowl', 'Cabinet'],
 'FloorPlan7': ['HousePlant', 'DiningTable'],
 'FloorPlan9': ['DiningTable']}

In [60]:
living_room_obj_dict = dict()
living_room_obj_per = {k:0 for k in living_room_objects}
total_obj=0
for l in living_rooms[:10]:
	controller.scene = l
	controller.reset()
	obj_set = [obj["objectType"] for obj in controller.last_event.metadata["objects"] if obj["objectType"] in living_room_objects]
	if len(obj_set) > 0:
		obj_selected = list(set(obj_set))
		obj_keep=[]
		for o in obj_selected:
			if living_room_obj_per[o] < 5:
				living_room_obj_per[o]+=1
				obj_keep.append(o)
				
		obj_selected = obj_keep
		#kitchen_objects.remove(obj_selected)
		living_room_obj_dict[l]=obj_selected
		total_obj+=len(obj_selected)
living_room_obj_dict, total_obj

({'FloorPlan201': ['HousePlant', 'Bowl', 'CoffeeTable', 'KeyChain'],
  'FloorPlan202': ['HousePlant', 'CoffeeTable', 'KeyChain'],
  'FloorPlan203': ['HousePlant',
   'CoffeeTable',
   'Bowl',
   'KeyChain',
   'WateringCan'],
  'FloorPlan204': ['HousePlant', 'WateringCan', 'CoffeeTable', 'KeyChain'],
  'FloorPlan205': ['HousePlant', 'WateringCan', 'KeyChain'],
  'FloorPlan206': ['Bowl', 'CoffeeTable'],
  'FloorPlan207': [],
  'FloorPlan208': [],
  'FloorPlan209': ['WateringCan'],
  'FloorPlan210': ['WateringCan']},
 23)

In [61]:
living_rooms_obj_dict=dict()
for l in living_rooms[:-10]:
	controller.scene = l
	controller.reset()
	obj_set = [obj["objectType"] for obj in controller.last_event.metadata["objects"] if obj["objectType"] in living_room_objects]
	if len(obj_set) > 0:
		obj_selected = obj_set[0]
		living_room_objects.remove(obj_selected)
		living_rooms_obj_dict[l] = obj_selected
living_rooms_obj_dict



{'FloorPlan201': 'Bowl',
 'FloorPlan202': 'CoffeeTable',
 'FloorPlan203': 'HousePlant',
 'FloorPlan204': 'KeyChain',
 'FloorPlan205': 'WateringCan'}

In [62]:
bedrooms_obj_dict = dict()
for b in bedrooms[:-10]:
	controller.scene = b
	controller.reset()
	obj_set = [obj["objectType"] for obj in controller.last_event.metadata["objects"] if obj["objectType"] in bedroom_objects]
	if len(obj_set) > 0:
		obj_selected = obj_set[0]
		bedroom_objects.remove(obj_selected)
		bedrooms_obj_dict[b] = obj_selected

bathrooms_obj_dict = dict()
for b in bathrooms[:-10]:
	controller.scene = b
	controller.reset()
	obj_set = [obj["objectType"] for obj in controller.last_event.metadata["objects"] if obj["objectType"] in bathroom_objects]
	if len(obj_set) > 0:
		obj_selected = obj_set[0]
		bathroom_objects.remove(obj_selected)
		bathrooms_obj_dict[b]=obj_selected

bedrooms_obj_dict, bathrooms_obj_dict

({'FloorPlan301': 'Bed',
  'FloorPlan302': 'Book',
  'FloorPlan303': 'Chair',
  'FloorPlan304': 'Bowl',
  'FloorPlan305': 'LightSwitch',
  'FloorPlan306': 'SideTable',
  'FloorPlan307': 'Painting'},
 {'FloorPlan401': 'Candle',
  'FloorPlan402': 'Cabinet',
  'FloorPlan403': 'LightSwitch',
  'FloorPlan404': 'Sink',
  'FloorPlan405': 'SprayBottle',
  'FloorPlan406': 'Toilet'})

In [63]:
kitchens_obj_dict

{'FloorPlan1': ['Knife', 'Sink', 'Fridge', 'HousePlant', 'Bowl', 'Cabinet'],
 'FloorPlan2': ['Knife', 'Sink', 'Fridge', 'Bowl', 'Cabinet'],
 'FloorPlan3': ['Knife', 'Sink', 'Fridge', 'HousePlant', 'Bowl', 'Cabinet'],
 'FloorPlan4': ['Knife',
  'Sink',
  'Fridge',
  'HousePlant',
  'Bowl',
  'Cabinet',
  'DiningTable'],
 'FloorPlan5': ['Knife', 'Sink', 'Fridge', 'HousePlant', 'Bowl', 'Cabinet'],
 'FloorPlan7': ['HousePlant', 'DiningTable'],
 'FloorPlan9': ['DiningTable']}

In [ ]:
# get goal coordinates

positions = controller.step(
			action="GetReachablePositions"
		).metadata["actionReturn"]

obj_list = [o for o in controller.last_event.metadata["objects"] if o["objectType"] == "Bowl"]
print(obj_list)
obj_pos = obj_list[0]["position"]
obj_pos = np.array([obj_pos["x"], obj_pos["y"], obj_pos["z"]])

positions_array = np.array([[p["x"], p["y"], p["z"]] for p in positions])
distance_per_array = np.linalg.norm(positions_array[:, ::2] - obj_pos[::2], axis=1)
positions_array_l=[dict(x=pos[0], y=pos[1], z=pos[2]) for pos in positions_array[distance_per_array<2.5].tolist()]
i_position = np.random.choice(positions_array_l)
print(i_position)


[]


IndexError: list index out of range

In [33]:

controller.stop()

In [30]:
e=controller.reset()

In [18]:
living_rooms_obj_dict

{'FloorPlan201': 'Bowl',
 'FloorPlan202': 'CoffeeTable',
 'FloorPlan203': 'HousePlant',
 'FloorPlan204': 'KeyChain',
 'FloorPlan205': 'WateringCan'}

In [37]:
action

{'x': -3.25, 'y': 0.9026566743850708, 'z': 4.5}

In [49]:
controller.reset()

<ai2thor.server.Event at 0x306f457e0
    .metadata["lastAction"] = Initialize
    .metadata["lastActionSuccess"] = True
    .metadata["errorMessage"] = "
    .metadata["actionReturn"] = {'cameraNearPlane': 0.009999999776482582, 'cameraFarPlane': 20.0}
>

['Cloth']

In [1]:
living_rooms

NameError: name 'living_rooms' is not defined

In [16]:
living_room_obj_dict={l:[] for l in living_rooms[:-10]}


In [17]:
living_room_objects = ["HousePlant", "CoffeeTable", "Cloth", "KeyChain", "WateringCan", "Bowl"]

for l in living_rooms[:-10]:
	controller.scene = l
	controller.reset()
	for obj in living_room_objects:
		if obj in [o["objectType"] for o in controller.last_event.metadata["objects"]]:
			living_room_obj_dict[l].append(obj)

In [18]:
living_room_obj_dict

{'FloorPlan201': ['HousePlant', 'CoffeeTable', 'KeyChain', 'Bowl'],
 'FloorPlan202': ['HousePlant', 'CoffeeTable', 'KeyChain'],
 'FloorPlan203': ['HousePlant',
  'CoffeeTable',
  'KeyChain',
  'WateringCan',
  'Bowl'],
 'FloorPlan204': ['HousePlant', 'CoffeeTable', 'KeyChain', 'WateringCan'],
 'FloorPlan205': ['HousePlant', 'KeyChain', 'WateringCan'],
 'FloorPlan206': ['HousePlant', 'CoffeeTable', 'KeyChain', 'Bowl'],
 'FloorPlan207': ['HousePlant', 'CoffeeTable', 'KeyChain'],
 'FloorPlan208': ['HousePlant', 'KeyChain'],
 'FloorPlan209': ['HousePlant', 'CoffeeTable', 'KeyChain', 'WateringCan'],
 'FloorPlan210': ['HousePlant', 'KeyChain', 'WateringCan']}

In [ ]:
import shutil
# Define the base directories
sceneType="living_room"
temp_folder = "temp_dataset/raw"
final_folder = f"test_dataset/{sceneType}"

# Ensure the temporary folder exists
os.makedirs(temp_folder, exist_ok=True)

In [71]:
# Clean the folders from the files inside
def clean_folder(path):
	for filename in os.listdir(path):
		file_path = os.path.join(path, filename)
		if os.path.isfile(file_path):
			os.remove(file_path)

sure = input("Are you sure you want to clean the folders? (Scenetype: {})".format(sceneType))

if "y" in sure.lower():
	clean_folder("temp_dataset")
	clean_folder("temp_dataset/raw")
	clean_folder(f"test_dataset/{sceneType}")
	clean_folder(f"test_dataset/{sceneType}/raw")

	

In [45]:
from thortils.agent import thor_agent_position, thor_agent_pose

In [73]:
run=0
ts=0
controller.reset()
tmp_save_dict = {
	"run": [],
	"goal": [],
	"sceneType": [],
	"answer": [],
	"expl": [],
	"compass": [],
	"object": [],
	"ts": []
}

real_save_dict = {
	"run": [],
	"goal": [],
	"sceneType": [],
	"answer": [],
	"expl": [],
	"compass": [],
	"object": [],
	"ts": []
}

#for d in [kitchens_obj_dict, living_rooms_obj_dict, bedrooms_obj_dict, bathrooms_obj_dict]:
for d in [living_room_obj_dict]:
	sceneType="living_room"
	for scene, goals in d.items():
		
		for goal in goals:
			tmp_save_dict = {
				"run": [],
				"goal": [],
				"sceneType": [],
				"answer": [],
				"expl": [],
				"compass": [],
				"object": [],
				"ts": []
			}
			print(f"Starting eval of episode {ts}, scene {scene} with target: {goal}")
			controller.scene = scene
			controller.reset()
			#thor_place_agent_randomly(controller, v_angles=[30])
			sleep(2)
			
			success=False
			n_steps=0
			
			i_s=list(range(1, num_points+1))
			compass=[[] for _ in range(24)]

			# Get closest object of type "goal"
			closest_id = None
			closest_dist = 1000
			obj_pos = None
			object_ids = []
			for o in controller.last_event.metadata["objects"]:
				if o["objectType"] == goal:
					if o["distance"] < closest_dist:
						closest_dist = o["distance"]
						closest_id = o["objectId"]
					object_ids.append(o["objectId"])

			#closest_id = np.random.choice(object_ids)
			if closest_id is None:
				print(f"Could not find object of type {goal}")
				continue

			found_plan = False
			max_tries = 10
			while not found_plan and max_tries>0:
				try:
					start_pos, start_rot = thor_agent_pose(controller, as_tuple=True)
					plan=get_shortest_path_to_object(controller, closest_id, start_position=start_pos, start_rotation=start_rot, return_plan=True)
					found_plan=True
				except BaseException as e:
					thor_place_agent_randomly(controller, v_angles=[30])
					
				
				closest_id = None
				closest_dist = 1000
				obj_pos = None
				object_ids = []
				for o in controller.last_event.metadata["objects"]:
					if o["objectType"] == goal:
						if o["distance"] < closest_dist:
							closest_dist = o["distance"]
							closest_id = o["objectId"]
						object_ids.append(o["objectId"])
				max_tries-=1
			
			if not found_plan:
				controller.reset()
				try:
					plan=get_shortest_path_to_object(controller, closest_id, start_position=start_pos, start_rotation=start_rot, return_plan=True)
				except BaseException as e:
					print(f"Could not find object of type {goal}")
					break
					

			for action in plan[1]:
				image=Image.fromarray(controller.last_event.frame)
				img=np.array(image)[:, :, ::-1].copy()

				cv2.imwrite(f"temp_dataset/raw/{ts}.png", img)

				for i, point in zip(i_s, ellipse_points):
						cv2.putText(img, str(i), (point[0]-7, point[1]-7), 2, 1, (0,255,255), 2)

				i_s=i_s+[8,9,0]
				# Display the result
				to_finish=False
				img_annotated = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
				
				visibleObjects=[]
				
				for o in controller.last_event.metadata["objects"]:
					if o["visible"] and o["objectType"] != "Floor":
						visibleObjects.append(o["objectType"])
				
				expl=""
				if "Move" in action[0]:
					answer="4"
				elif "Left" in action[0]:
					answer="6"
				elif "Right" in action[0]:
					answer="3"
				elif "Up" in action[0]:
					answer="8"
				elif "Down" in action[0]:
					answer="9"
				else:
					answer="0"
					to_finish=True
				
				# Save_everything:
				cv2.imwrite(f"temp_dataset/{ts}.png", img)
				tmp_save_dict["expl"].append(expl)
				tmp_save_dict["goal"].append(goal)
				tmp_save_dict["run"].append(run)
				tmp_save_dict["object"].append(list(set(visibleObjects)))
				tmp_save_dict["compass"].append(compass)
				tmp_save_dict["answer"].append(answer)
				tmp_save_dict["ts"].append(ts)

				ts+=1

				for i in i_s:
					if str(i) in answer:
						direction = i_s.index(i)
						for o in visibleObjects:
							if o not in compass[6] and o != "Floor" and o!="Wall":
								compass[6].append(o)
						if direction not in [7,8,9]:
							compass = rotate_compass(compass, direction)
						action_to_execute = actions[direction]

				if "Move" in action[0]:
					event = controller.step(action=action[0], moveMagnitude=action[1][0])
				elif "Rotate" in action[0]:
					event = controller.step(action=action[0], degrees=abs(action[1][1]))
				else:
					event = controller.step(action=action[0], degrees=abs(action[1][1]))

				sleep(1)
			# Check if the object is visible and distance is less than 1.5
			if event.metadata["lastActionSuccess"]:
				for o in event.metadata["objects"]:
					if o["objectId"] == closest_id:
						if o["visible"] and o["distance"] < 1.5:
							success=True
							run+=1
							break
						
			if success:
				# Save all the data with the same run number
				print(f"Finished episode {ts}")
				for k,v in tmp_save_dict.items():
					real_save_dict[k].extend(v)
					for filename in os.listdir("temp_dataset/raw"):
						shutil.move(os.path.join("temp_dataset/raw", filename), os.path.join("test_dataset", sceneType, "raw", filename))
					for filename in os.listdir("temp_dataset"):
						# if filename is not a folder
						if not os.path.isdir(os.path.join("temp_dataset", filename)):
							shutil.move(os.path.join("temp_dataset", filename), os.path.join("test_dataset", sceneType, filename))
					print(f"Successfully saved images to {final_folder}")
			else:
				# Delete tmp images
				clean_folder("temp_dataset")
				clean_folder("temp_dataset/raw")
				print("Failed to reach goal")
				

Starting eval of episode 0, scene FloorPlan201 with target: HousePlant
Failed to reach goal
Starting eval of episode 10, scene FloorPlan201 with target: CoffeeTable
Finished episode 30
Successfully saved images to test_dataset/living_room
Successfully saved images to test_dataset/living_room
Successfully saved images to test_dataset/living_room
Successfully saved images to test_dataset/living_room
Successfully saved images to test_dataset/living_room
Successfully saved images to test_dataset/living_room
Successfully saved images to test_dataset/living_room
Successfully saved images to test_dataset/living_room
Starting eval of episode 30, scene FloorPlan201 with target: KeyChain
Finished episode 50
Successfully saved images to test_dataset/living_room
Successfully saved images to test_dataset/living_room
Successfully saved images to test_dataset/living_room
Successfully saved images to test_dataset/living_room
Successfully saved images to test_dataset/living_room
Successfully saved imag

In [72]:
controller = init_controller()

In [37]:
controller.last_event.metadata["agent"]

{'name': 'agent',
 'position': {'x': -1.0, 'y': 0.900999128818512, 'z': 1.0},
 'rotation': {'x': -0.0, 'y': 270.0, 'z': 0.0},
 'cameraHorizon': 0.0,
 'isStanding': True,
 'inHighFrictionArea': False}

In [76]:
controller.stop()

In [74]:
real_save_dict["sceneType"]=[sceneType]*len(real_save_dict["run"])

In [75]:

if os.path.exists(f"test_dataset/{sceneType}/all.csv"):
	old_df = pd.read_csv(f"test_dataset/{sceneType}/all.csv", index_col=0)
	df=pd.DataFrame.from_dict(real_save_dict)
	pd.concat((old_df, df), axis=0).to_csv(f"test_dataset/{sceneType}/all.csv")
else:
	pd.DataFrame.from_dict(real_save_dict).to_csv(f"test_dataset/{sceneType}/all.csv")


In [51]:
controller.stop()

In [77]:
model_id="microsoft/swin-large-patch4-window7-224"
processor = AutoImageProcessor.from_pretrained(model_id)
s_model = SwinModel.from_pretrained(model_id).to("mps")


In [ ]:
df=pd.read_csv(f"test_dataset/{sceneType}/all.csv", index_col=0)

emb_array=[]

filenames = os.listdir(f"test_dataset/{sceneType}/raw")
# Order by name (str to int)
filenames.sort(key=lambda x: int(x.split(".")[0]))

for f in filenames:
	i=f.split(".")[0]
	img=Image.open(f"test_dataset/{sceneType}/raw/{i}.png")
	inputs = processor(img, return_tensors="pt")["pixel_values"].to("mps")
	with torch.no_grad():
		outputs = s_model(inputs)
	emb_array.append(outputs.last_hidden_state.mean(dim=1)[0].detach().cpu().numpy())
emb = torch.tensor(np.array(emb_array))
torch.save(emb, f"embeddings/embedding_{sceneType}.pt")

FileNotFoundError: [Errno 2] No such file or directory: '/Users/davidebuoso/Downloads/thortils/test_dataset/living_room/raw/0.png'

In [ ]:
controller.stop_unity()


In [67]:
bathrooms_obj_dict

{'FloorPlan401': 'Candle',
 'FloorPlan402': 'Cabinet',
 'FloorPlan403': 'LightSwitch',
 'FloorPlan404': 'Sink',
 'FloorPlan405': 'SprayBottle',
 'FloorPlan406': 'Toilet'}

kitchen_object